# Attention Mechanism:
## Keys, Queries, and Values


## How To Use This Notebook

Read each short explanation, then run the code cell below it.

Focus on these questions:

1. Which word is asking for context?
2. Which words match that question?
3. How are scores converted into weights?
4. How do weights mix the Value vectors?

## 1. Helper Functions

In [ ]:
import math


def dot(a, b):
    """Return the dot-product matching score for two equal-length vectors."""
    # In attention, Query and Key vectors are compared using dot product.
    return sum(x * y for x, y in zip(a, b))


def softmax(scores):
    """Convert raw scores into weights that add up to 1."""
    # Subtract the maximum score to keep exponentials stable.
    max_score = max(scores)
    exps = [math.exp(score - max_score) for score in scores]
    total = sum(exps)
    return [value / total for value in exps]


def weighted_sum(weights, vectors):
    """Mix vectors using the given weights and return one output vector."""
    # Each output dimension is the weighted sum of that dimension across vectors.
    return [
        sum(weight * vector[i] for weight, vector in zip(weights, vectors))
        for i in range(len(vectors[0]))
    ]


def fmt(x):
    """Format a number to two decimal places for readable tables."""
    return f"{x:.2f}"


def fmt_vec(vector):
    """Format a numeric vector as a short readable string."""
    return "[" + ", ".join(fmt(x) for x in vector) + "]"


def show_table(headers, rows):
    """Print a simple aligned text table without extra packages."""
    # Find the best width for each column.
    widths = [
        max(len(str(x)) for x in [header] + [row[i] for row in rows])
        for i, header in enumerate(headers)
    ]

    # Print table header.
    line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))

    # Print table body.
    for row in rows:
        print(" | ".join(str(x).ljust(widths[i]) for i, x in enumerate(row)))


def show_bars(words, weights):
    """Show attention weights as small text bars."""
    rows = []
    for word, weight in zip(words, weights):
        # A bigger weight creates a longer bar.
        bar = "#" * max(1, round(weight * 20)) if weight > 0 else "."
        rows.append([word, fmt(weight), bar])
    show_table(["Word", "Weight", "Visual"], rows)

## 2. From Text To Embeddings

Attention does not work on raw text.

First:

```text
text -> tokens -> token IDs -> embeddings
```

An embedding is a vector for a token.

In [ ]:
text = "I love NLP"

# Split raw text into tokens.
tokens = text.split()

# Small toy vocabulary: each token maps to one ID.
vocab = {"I": 2, "love": 3, "NLP": 4}
token_ids = [vocab[token] for token in tokens]

# Tiny invented embeddings for understanding.
# Real models learn these vectors from data.
embeddings = {
    2: [0.90, 0.10, 0.00],
    3: [0.20, 0.80, 0.30],
    4: [0.10, 0.20, 0.90],
}

# Display the full path: token -> ID -> vector.
rows = [
    [token, token_id, fmt_vec(embeddings[token_id])]
    for token, token_id in zip(tokens, token_ids)
]
show_table(["Token", "Token ID", "Embedding"], rows)

**Note**

Embeddings give each token a starting vector.

Attention updates that vector using context.

Example:

```text
"good" alone may look positive.
"not good" is negative because "not" changes the meaning.
```

## 3. Attention As Weights

For one query word, attention creates one row of weights.

The weights say:

```text
how much should I look at each word?
```

The weights add up to 1.

In [ ]:
words = ["The", "battery", "life", "is", "amazing", "but", "screen", "dull"]

# Example attention row for the topic "battery life".
# Important words get larger weights.
weights = [0.02, 0.30, 0.25, 0.03, 0.35, 0.01, 0.02, 0.02]

show_bars(words, weights)
print("Total weight:", fmt(sum(weights)))

## 4. Query, Key, Value

Use this simple meaning:

| Term | Meaning |
|---|---|
| Query | what I am looking for |
| Key | what I compare against |
| Value | information I will use |

Important:

```text
Query + Key decide the weight.
Value gives the information to mix.
```

In [ ]:
x_good = [0.40, 0.90, -0.20]

# Same starting embedding, shown in three attention roles.
q_good = [0.85, 0.57]        # Query: what "good" looks for
k_good = [0.37, 0.98]        # Key: how "good" can be matched
v_good = [0.40, 0.75, -0.03] # Value: information "good" contributes

show_table(
    ["Vector", "Value", "Meaning"],
    [
        ["Embedding", fmt_vec(x_good), "starting vector"],
        ["Query", fmt_vec(q_good), "what good looks for"],
        ["Key", fmt_vec(k_good), "how good is matched"],
        ["Value", fmt_vec(v_good), "what good contributes"],
    ],
)

**Example note**

Query and Key must have the same size because we compare them using dot product.

Value can have a different size because it carries information, not matching score.

## 5. Attention Formula In Simple Words

Formula:

```text
Attention(Q, K, V) = softmax((QK^T) / sqrt(d_k)) V
```

Read it as:

```text
compare -> scale -> softmax -> combine
```

Meaning:

| Part | Meaning |
|---|---|
| `QK^T` | compare Queries with Keys |
| `/ sqrt(d_k)` | keep scores from becoming too large |
| `softmax` | turn scores into weights |
| `V` | mix Value information |

## 6. Worked Example: `"not good"`

Sentence:

```text
The movie was not good.
```

Focus word:

```text
good
```

We will use only three words to keep the math small:

```text
movie, not, good
```

In [ ]:
focus_words = ["movie", "not", "good"]

# Query from the focus word "good".
q_good = [1, 1]

# Keys from candidate words.
keys = {
    "movie": [1, 0],
    "not": [2, 2],
    "good": [1, 2],
}

# 1. Match: query dot each key.
raw_scores = [dot(q_good, keys[word]) for word in focus_words]

# 2. Scale: divide by sqrt(d_k). Here d_k = 2.
scaled_scores = [score / math.sqrt(2) for score in raw_scores]

# 3. Weight: softmax makes weights add to 1.
attention_weights = softmax(scaled_scores)

rows = [
    [word, keys[word], fmt(raw), fmt(scaled), fmt(weight)]
    for word, raw, scaled, weight in zip(
        focus_words, raw_scores, scaled_scores, attention_weights
    )
]
show_table(["Word", "Key", "Raw score", "Scaled", "Weight"], rows)

print()
show_bars(focus_words, attention_weights)

**Note**

The word `"not"` receives the highest weight.

That means `"good"` is strongly updated using information from `"not"`.

## 7. Combine Values

Attention weights are not the final answer.

They tell us how much of each Value vector to use.

For easy understanding, use human-named dimensions:

```text
[movie-topic, negation-signal, positive-word]
```

In [ ]:
# Value vectors use named dimensions only for easy interpretation:
# [movie-topic, negation-signal, positive-word]
values = {
    "movie": [1, 0, 0],
    "not": [0, 1, 0],
    "good": [0, 0, 1],
}

# Mix values using the attention weights from the previous cell.
output_good = weighted_sum(
    attention_weights,
    [values[word] for word in focus_words],
)

show_table(
    ["Output dimension", "Amount"],
    [
        ["movie-topic", fmt(output_good[0])],
        ["negation-signal", fmt(output_good[1])],
        ["positive-word", fmt(output_good[2])],
    ],
)

**Example note**

The updated vector for `"good"` now contains strong negation information.

This is why attention helps the model understand `"not good"`.

## 8. Self-Attention Matrix

In self-attention, every word can create its own attention row.

For `T` tokens, the matrix size is:

```text
T x T
```

Rows are query words.
Columns are words being looked at.

In [ ]:
matrix_words = ["The", "movie", "was", "not", "good"]

# Already scaled scores for a compact example.
# Each row belongs to one query word.
score_matrix = [
    [3.0, 1.1, 1.0, 0.4, 0.4],
    [0.7, 2.7, 1.2, 0.3, 1.0],
    [1.0, 1.0, 2.0, 0.8, 0.8],
    [0.2, 0.3, 0.3, 2.8, 1.8],
    [0.2, 0.7, 0.2, 3.0, 1.9],
]

# Apply softmax separately to every row.
attention_matrix = [softmax(row) for row in score_matrix]

rows = []
for query_word, weights_row in zip(matrix_words, attention_matrix):
    rows.append([query_word] + [fmt(w) for w in weights_row])

show_table(["Query"] + matrix_words, rows)

## 9. Masks

Masks control which positions attention is allowed to use.

Two important masks:

| Mask | Purpose |
|---|---|
| Padding mask | block fake `PAD` tokens |
| Causal mask | block future tokens |

In [ ]:
tokens = ["I", "love", "deep", "learning"]

# Causal mask for GPT-style next-word prediction.
# A token can look at itself and previous tokens only.
causal_mask = [
    [1 if col <= row else 0 for col in range(len(tokens))]
    for row in range(len(tokens))
]

show_table(["Query"] + tokens, [[tok] + row for tok, row in zip(tokens, causal_mask)])

**Note**

In a causal mask:

```text
1 = can look
0 = blocked
```

This prevents the model from seeing future words during next-word prediction.

## 10. Cross-Attention

Self-attention:

```text
words look at words in the same sentence
```

Cross-attention:

```text
one sequence looks at another sequence
```

Translation example:

```text
pommes should look strongly at apples
```

In [ ]:
english_words = ["I", "love", "apples"]

# When generating "pommes", cross-attention should focus on "apples".
weights_for_pommes = softmax([0.3, 1.0, 3.2])

show_bars(english_words, weights_for_pommes)

## 11. Multi-Head Attention

One attention head gives one way of looking.

Multi-head attention gives several ways of looking at the same sentence.

Example:

```text
Head 1 may focus on negation.
Head 2 may focus on the topic.
Head 3 may focus on contrast words like "but".
```

In [ ]:
words = ["movie", "not", "good", "but", "acting", "excellent"]

# Three heads, three different attention patterns.
heads = {
    "negation": [0.10, 0.60, 0.20, 0.05, 0.02, 0.03],
    "topic": [0.55, 0.10, 0.25, 0.04, 0.03, 0.03],
    "contrast": [0.10, 0.10, 0.25, 0.40, 0.05, 0.10],
}

for head_name, head_weights in heads.items():
    print("\nHead:", head_name)
    show_bars(words, head_weights)

## 12. Practice

Sentence:

```text
The food was not tasty but the service was excellent.
```

If the query word is `"tasty"`, useful context should include:

```text
food, not, tasty
```

In [ ]:
practice_words = ["food", "not", "tasty", "service", "excellent"]

# Scores are chosen so "not" receives the highest attention for "tasty".
tasty_scores = [1.2, 2.8, 1.5, 0.2, 0.1]
tasty_weights = softmax(tasty_scores)

show_bars(practice_words, tasty_weights)

## 13. Common Doubts

**Doubt 1: Are Key and Value the same?**

No.

```text
Key = used for matching
Value = information that gets mixed
```

**Doubt 2: Does attention choose only one word?**

Usually no. It gives weights to many words.

**Doubt 3: What is `d_k`?**

`d_k` is the size of Query and Key vectors.

Example:

```text
q_good = [1, 1]
d_k = 2
```